# DAIC-WOZ: reliability-aware knowledge distillation
## Notebook 01 — setup and data audit
Run in Google Colab. This starter inspects file structure and official split metadata; it does **not** train a model or claim experimental results.

Workflow: data audit → audio/text teachers → compact student → standard KD → reliability-aware KD → participant-level evaluation.

Keep licensed recordings, transcripts, labels, features, checkpoints, and participant predictions on your private Drive. Commit only source code and cleared notebooks to GitHub. Clear all notebook outputs before saving a copy to GitHub.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, platform, sys
import pandas as pd

drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/DAIC_WOZ')  # edit if needed
RUN_ROOT = Path('/content/drive/MyDrive/DAIC_WOZ_experiments')
assert DATA_ROOT.is_dir(), f'Update DATA_ROOT: {DATA_ROOT} does not exist'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], '| Platform:', platform.platform())
print('Dataset folder found.')


### 1. Inspect names only
No audio or transcript contents are read. Archives are listed but are not extracted. If the data are archived, share the archive/folder layout before building preprocessing.


In [ ]:
items = sorted(DATA_ROOT.iterdir())
for p in items[:50]:
    print('[DIR]' if p.is_dir() else '[FILE]', p.name)
folders = [p for p in items if p.is_dir()]
if folders:
    sample = next((p for p in folders if p.name.endswith('_P')), folders[0])
    print('\nExample folder:', sample.name)
    for p in sorted(sample.iterdir())[:30]:
        print(' ', p.name)


### 2. Locate split CSVs
Select the official participant split CSVs. Training and development labels are required for later training. Test labels may be unavailable and are not required for setup. Filenames below are suggestions only: verify them against your dataset release. Never generate random segment-level train/test splits.


In [ ]:
csv_paths = sorted(DATA_ROOT.rglob('*.csv'))
candidates = [p for p in csv_paths if any(t in p.name.lower() for t in ('split', 'train', 'dev', 'test'))]
for p in candidates[:60]:
    print(p.relative_to(DATA_ROOT))
print('Candidate CSV count:', len(candidates))


In [ ]:
# Set paths relative to DATA_ROOT using the listing above.
# Leave None until you have identified the correct official CSVs.
SPLIT_FILES = {'train': None, 'dev': None, 'test': None}
# Example only: SPLIT_FILES['train'] = 'train_split_Depression_AVEC2017.csv'

split_ids, audit = {}, {}
for split, relative in SPLIT_FILES.items():
    if relative is None:
        print(split, ': not configured')
        continue
    path = (DATA_ROOT / relative).resolve()
    assert path.is_relative_to(DATA_ROOT.resolve()), 'Split must be inside DATA_ROOT'
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    id_col = next((c for c in df.columns if c.lower() in
                   ('participant_id', 'participant', 'subject_id')), None)
    assert id_col is not None, f'No participant ID column in {relative}: {list(df.columns)}'
    ids = pd.to_numeric(df[id_col], errors='raise')
    assert ids.notna().all() and (ids % 1 == 0).all(), 'Invalid participant IDs'
    assert not ids.duplicated().any(), f'Duplicate participant rows in {split}'
    split_ids[split] = set(ids.astype(int))
    audit[split] = {'participants': len(ids), 'columns': list(df.columns)}
    print(split, ':', len(ids), 'participants; columns:', list(df.columns))

from itertools import combinations
for a, b in combinations(split_ids, 2):
    overlap = split_ids[a] & split_ids[b]
    assert not overlap, f'LEAKAGE: {len(overlap)} overlapping participants in {a}/{b}'
    print(a, '/', b, ': no participant overlap')

if len(split_ids) < 2:
    print('Audit incomplete: configure at least train and dev CSVs.')
else:
    print('Configured split metadata passed overlap checks. Segment and preprocessing audits remain.')


### 3. Save a private setup report
The report contains counts, column names and configuration status, not transcript text or participant IDs. A successful metadata audit is not proof that future preprocessing or teacher checkpoints are leakage-free.


In [ ]:
report = {
    'stage': 'setup_only_no_training',
    'splits': audit,
    'configured_splits': sorted(split_ids),
    'missing_splits': sorted(set(SPLIT_FILES) - set(split_ids)),
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
}
report_path = RUN_ROOT / 'setup_audit.json'
report_path.write_text(json.dumps(report, indent=2))
print('Saved setup_audit.json in your experiment folder.')


## Experiment rules for the next notebook
- Keep every participant's segments in one split; aggregate predictions per participant.
- Fit normalization and feature selection on the training partition only.
- Develop teachers and the student using participant-grouped training folds. Calibrate reliability using held-out predictions rather than teacher training accuracy.
- Compare the same student with no KD, fixed-weight KD, and reliability-aware KD under matched training budgets.
- Select hyperparameters and decision thresholds without test feedback. Predeclare seeds and report all runs, not the best seed.
- Evaluate clean inputs and specified missing/noisy-modality conditions, with macro-F1, depressed-class F1, balanced accuracy and AUROC where defined. Use participant-level confidence intervals.
- Verify that any imported teacher checkpoint never trained on evaluation participants.

**Next input needed:** folder listing and split CSV column names from this notebook. No raw participant data need to be shared. These determine the actual audio/transcript loader and training notebook.

## GitHub / Colab workflow
Keep this file at `notebooks/01_daic_woz_setup.ipynb`. Open it from Colab's GitHub tab after granting access to the private repository. After running, clear outputs before using **File → Save a copy in GitHub**. Dataset files and experiment outputs remain in Drive.
